# 02 — Task 1: Logistic Regression *from scratch*

**Hard constraint:** no `sklearn.LogisticRegression` or any pre-built logistic regression — using one scores 0. Implement `sigmoid`, `loss`, `gradients`, `train`, `predict` by hand. Deliverable: `submissions/LogReg_predictions.csv`.

## 0. Setup

Imports, src helpers, paths.

In [1]:
# Adds project root to path so `import src...` works from notebooks/.
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src import paths, data, evaluation


## 1. Load features + the locked split

Reuse the saved indices — never re-split.

In [2]:
X, y, ids = data.load_train_features()
dev_idx = np.load(paths.DATA_PROCESSED / 'dev_idx.npy')
holdout_idx = np.load(paths.DATA_PROCESSED / 'holdout_idx.npy')

## 2. The model — required functions

Signatures fixed by the brief.

In [3]:
def sigmoid(z):
    """Numerically stable logistic sigmoid."""
    z = np.asarray(z, dtype=float)
    # Calculate negative and positive z separately to prevent floating-point errors
    result = np.empty_like(z)
    positive = z >= 0
    result[positive] = 1 / (1 + np.exp(-z[positive]))
    exp_z = np.exp(z[~positive])
    result[~positive] = exp_z / (1 + exp_z)
    return result.item() if result.ndim == 0 else result


def loss(y, y_hat):
    """Binary log loss."""
    # Clip to prevent log(0) or log(1)
    y_hat = np.clip(y_hat, np.finfo(float).eps, 1 - np.finfo(float).eps)
    m = y.shape[0]
    inner = np.sum(y * np.log(y_hat) + (1-y) * np.log(1 - y_hat))
    return -inner/m


def gradients(X, y, y_hat):
    """Return (dw, db)."""
    m = y.shape[0]
    dw = np.matmul(X.T, y_hat-y)/m
    db = np.sum(y_hat - y)/m
    return (dw, db)


def train(X, y, bs, epochs, lr):
    """Mini-batch gradient descent. Returns learned (w, b) and loss history."""
    loss_history = []
    b = 0
    w = np.ones(X.shape[1])
    y_hat = sigmoid(np.matmul(w, X.T) + b)
    loss_history.append(loss(y, y_hat))
    for i in range(epochs):
        for start in range(0, X.shape[0], bs):
            X_batch = X[start:start + bs]
            y_batch = y[start:start + bs]
            y_hat = sigmoid(np.matmul(w, X_batch.T) + b)
            dw, db = gradients(X_batch, y_batch, y_hat)
            w = w - lr * dw
            b = b - lr * db
        y_hat = sigmoid(np.matmul(w, X.T) + b)
        loss_history.append(loss(y, y_hat))
    return w, b, loss_history


def predict(X, w, b, threshold=0.5):
    """Return 0/1 labels."""
    return np.where(sigmoid(np.matmul(w, X.T) + b) >= threshold, 1, 0)

## 3. Train + validate

Tune the knobs (`bs`, `epochs`, `lr`); target performance comparable to sklearn LogReg.

In [4]:
w, b, hist = train(X[dev_idx], y[dev_idx], bs=4, epochs=300, lr=0.1)
print(evaluation.macro_f1(y[holdout_idx], predict(X[holdout_idx], w, b)))
hist[-10:]

0.7287862106572897


[np.float64(0.3142498745497136),
 np.float64(0.3140701632199407),
 np.float64(0.31389122256168694),
 np.float64(0.3137130467574791),
 np.float64(0.3135356300533409),
 np.float64(0.3133589667578841),
 np.float64(0.3131830512414142),
 np.float64(0.3130078779350532),
 np.float64(0.3128334413298778),
 np.float64(0.3126597359760714)]

## 4. Predict test + write submission

In [5]:
Xt, test_ids = data.load_test_features()
preds = predict(Xt, w, b)
data.write_submission(test_ids, preds, 'LogReg_predictions.csv')

## Discussion / carry-forward → `03_pca_knn.ipynb`

- Chosen values: bs=4, epochs=300, lr=0.1
- Holdout Macro F1: 0.7287862106572897